# reach-asr on Kaggle - the run behind the reported WER

The exact notebook that produced the numbers in the README, now extended with
the fourth cell of the 2x2.

|                | zero-shot | fine-tuned |
|----------------|-----------|------------|
| **clean**      | 4.37%     | *this run* |
| **degraded**   | 23.76%    | 21.20%     |

The first three were measured. The fourth - clean audio through the fine-tuned
model - was not, and it is the only one that separates two very different
outcomes:

- **"it learned to handle phone audio"** - clean/fine-tuned stays near 4.37%.
- **"it learned to ONLY handle phone audio"** - clean/fine-tuned is much worse,
  and the 2.56-point degraded gain was bought by giving up wideband speech.

A LoRA trained exclusively on one narrow degraded channel really can specialise
that way, and at low rank it is a routine outcome rather than an exotic one.
It costs one extra generation pass and no retraining.

Kept in the repo because `results/` is gitignored, so this is the only committed
record that the run happened and what it printed.

**Run it with Save Version -> Save & Run All (Commit), not interactively.** An
interactive session loses everything in `/kaggle/working` on a disconnect or an
idle timeout; this pipeline is ~60 minutes. Settings: Accelerator **GPU T4 x2**,
Internet **On** (both corpora stream from Hugging Face at runtime).


## 1. Environment and code

`torchao` is uninstalled deliberately: Kaggle ships 0.10.0, and PEFT's
`is_torchao_available()` *raises* on an incompatible version rather than
returning False — training dies at `get_peft_model()` before step 1. Nothing
here uses torchao.

The `rglob` finds the repo root at whatever depth the uploaded zip nested it,
instead of guessing the mount path.


In [1]:
!pip install -q jiwer peft
!pip uninstall -y -q torchao
import os
import shutil
import pathlib

hits = list(pathlib.Path("/kaggle/input").rglob("run_kaggle.py"))
if not hits:
    raise SystemExit("dataset not attached")
root = hits[0].parent.parent
for item in root.iterdir():
    dst = pathlib.Path("/kaggle/working") / item.name
    if dst.is_dir():
        shutil.rmtree(dst)
    elif dst.exists():
        dst.unlink()
    if item.is_dir():
        shutil.copytree(item, dst)
    else:
        shutil.copy2(item, dst)
os.chdir("/kaggle/working")
print(sorted(p.name for p in pathlib.Path(".").iterdir()))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 53.1 MB/s eta 0:00:00
['README.md', '__notebook__.ipynb', 'kaggle', 'reach_asr', 'tests']


## 2. Build the degraded corpus

`--snr-min -5 --snr-max 10 --packet-loss 0.05` is harsher than the defaults
(5–20 dB, 2%). That was chosen from a 7-minute headroom probe: `whisper-base`
showed a 19.2-point clean-vs-degraded gap here, against only 2.5 points for
`whisper-small` on the default channel — which is why the first attempt had
nothing to recover and made WER worse.


In [2]:
%cd /kaggle/working
!python -m reach_asr.build_dataset --train 2000 --eval 300 --snr-min -5 --snr-max 10 --packet-loss 0.05

/kaggle/working
loading ESC-50 noise bank (target 180 clips)...
README.md: 100%|███████████████████████████████| 345/345 [00:00<00:00, 1.54MB/s]
Repo card metadata block was not found. Setting CardData to empty.
dataset_infos.json: 1.61kB [00:00, 5.61MB/s]
data/train-00000-of-00002-2f1ab7b824ec75(…): 100%|█| 387M/387M [00:04<00:00, 80.
data/train-00001-of-00002-27425e5c1846b4(…): 100%|█| 387M/387M [00:03<00:00, 101
Generating train split: 100%|███████| 2000/2000 [00:03<00:00, 504.32 examples/s]
  180 noise clips across 15 categories
README.md: 11.0kB [00:00, 28.9MB/s]
Resolving data files: 100%|██████████████████| 48/48 [00:00<00:00, 29262.59it/s]
  train: 100/2000
  train: 200/2000
  train: 300/2000
  train: 400/2000
  train: 500/2000
  train: 600/2000
  train: 700/2000
  train: 800/2000
  train: 900/2000
  train: 1000/2000
  train: 1100/2000
  train: 1200/2000
  train: 1300/2000
  train: 1400/2000
  train: 1500/2000
  train: 1600/2000
  train: 1700/2000
  train: 1800/2000
  train: 19

## 3. LoRA fine-tune

`whisper-base`, 2 epochs, batch 16. 126 steps, ~6.5 minutes on the T4.

Targets are normalised through `EnglishTextNormalizer` — the same one WER is
scored with. Training on LibriSpeech's raw ALL-CAPS references teaches a
formatting change rather than acoustics, which is what doubled WER on the first
run while the loss curve fell the whole way.


In [3]:
%cd /kaggle/working
!python -m reach_asr.train --model openai/whisper-base --epochs 2 --batch-size 16 --grad-accum 1

/kaggle/working
device: cuda: Tesla T4
preprocessor_config.json: 185kB [00:00, 78.3MB/s]
config.json: 1.98kB [00:00, 6.32MB/s]
tokenizer_config.json: 283kB [00:00, 223MB/s]
vocab.json: 836kB [00:00, 32.4MB/s]
tokenizer.json: 2.48MB [00:00, 154MB/s]
merges.txt: 494kB [00:00, 105MB/s]
normalizer.json: 52.7kB [00:00, 89.3MB/s]
added_tokens.json: 34.6kB [00:00, 83.5MB/s]
special_tokens_map.json: 2.19kB [00:00, 9.52MB/s]
model.safetensors: 100%|██████████████████████| 290M/290M [00:02<00:00, 120MB/s]
Loading weights: 100%|█| 245/245 [00:00<00:00, 1647.92it/s, Materializing param=
generation_config.json: 3.81kB [00:00, 13.7MB/s]
trainable params: 1,179,648 || all params: 73,773,568 || trainable%: 1.5990
train utterances: 2000
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
  0%|                                                   | 0/126 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather a

## 4. The 2x2, with intervals

Four passes over the same 300 utterances: clean and degraded audio, each through
the stock model and the fine-tuned one.

Two comparisons come out of it. **Degraded zero-shot vs degraded fine-tuned** is
the result. **Clean zero-shot vs clean fine-tuned** is the specialisation check -
whether the model kept the ability it started with.

Both get a paired bootstrap confidence interval. Paired because every pass scores
the same utterances, and for the clean pair it is literally the same audio file
through two models, so resampling them independently would widen the interval
with variance the design already removed.

> **The stored output below is the original three-pass run** - it is the
> committed provenance for 4.37 / 23.76 / 21.20 and is left untouched. Re-running
> this cell now prints four passes and both intervals; the fourth number does not
> exist yet.


In [4]:
%cd /kaggle/working
!python -m reach_asr.evaluate_wer --model openai/whisper-base --adapter runs/whisper-lora/adapter


/kaggle/working
eval utterances: 300  device: cuda
Loading weights: 100%|█| 245/245 [00:00<00:00, 1109.20it/s, Materializing param=
[1/3] zero-shot on CLEAN audio (the ceiling)...
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor

## 5. Re-analysis, and filling in one cell later

`analyze.py` re-derives everything above from `results/predictions.jsonl` on CPU
in seconds - the intervals, the SNR breakdown with edges taken from this run's
own range, and a breakdown by noise category. No model load, no GPU. Use it to
re-read a finished run without paying for the generation passes again.


In [ ]:
%cd /kaggle/working
!python -m reach_asr.analyze --predictions results/predictions.jsonl --out results/analysis.json


### If you already have a finished run

Attach the previous version's output as a dataset input (so `data/` and
`runs/whisper-lora/adapter` are present) and run only the missing pass. A subset
of `--passes` **merges** into the existing `results/wer.json` rather than
overwriting it, so the three numbers that already cost half an hour of GPU
survive:

```
!python -m reach_asr.evaluate_wer --model openai/whisper-base     --adapter runs/whisper-lora/adapter --passes clean_finetuned
```

One generation pass instead of four.
